# Environment Setup

In this section, we import the necessary libraries, connect to google drive and configure the hardware acceleration.

In [ ]:
from google.colab import drive
import os
from pathlib import Path
from tqdm import tqdm
import random
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from numpy.lib.stride_tricks import sliding_window_view
from scipy.io import wavfile
from scipy.signal import resample_poly, convolve, firwin, filtfilt, spectrogram, get_window, fftconvolve
from multiprocessing import Pool, cpu_count
import random
import argparse

In [ ]:
from utils.args import args_parser
from utils.metrics import calc_nm_score, calc_cc_score, compute_top_k
from models.linear_fusion import LinearFusion
from models.encoder import TwoStageTCNEncoder
from pipelines.train_encoder import train_encoder
from pipelines.train_fusion import train_fusion_length_bucketed
from pipelines.dataset_encoder import BucketedBatchLoader, process_directory_parallel

In [ ]:
# Mount Google Drive to access the dataset and save models
drive.mount('/content/drive')

# Set working directory to the specific project folder
os.chdir('/content/drive/MyDrive/University/ENF/')
project_dir = Path.cwd()

# Configure Device: Use GPU (CUDA) if available for faster training, otherwise CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
args = args_parser(project_dir, device)

____________________________________________
# Train Encoder

## Dataset

### Functions

In [ ]:
def h_args_parser(harmonic, f_center, args):
    window_size = int(args['window_size_sec'] * args['fs'])
    h_args = {
        'harmonic': harmonic,
        'f_center': f_center,
        'window': get_window(args['window_type'], window_size),
        'noverlap': int(args['overlap_ratio'] * window_size),
        'nfft': 100 * window_size,
        'cutoff': [(f_center - args['f_range'] / 2) / (args['fs'] / 2), (f_center + args['f_range'] / 2) / (args['fs'] / 2)]
    }
    h_args['filter_coeff'] = firwin(args['band_pass_order'], h_args['cutoff'], pass_zero=False, window=args['window_type'])

    return h_args

def extract_enf(signal, args, h_args):
    signal = filtfilt(h_args['filter_coeff'], [1], signal)

    # Compute STFT
    f, t, s = spectrogram(
        signal,
        fs=args['fs'],
        window=h_args['window'],
        noverlap=h_args['noverlap'],
        nfft=h_args['nfft'],
        mode='complex'
    )

    # Define frequency range of interest
    f_low = h_args['f_center'] - args['f_range'] * h_args['harmonic'] / 2
    f_high = h_args['f_center'] + args['f_range'] * h_args['harmonic'] / 2

    # Find indices corresponding to the frequency range
    f1_idx = np.argmin(np.abs(f - f_low))
    f2_idx = np.argmin(np.abs(f - f_high))

    # Select frequency range and corresponding spectrogram values
    f_cut = f[f1_idx:f2_idx + 1]
    s_cut = np.abs(s[f1_idx:f2_idx + 1, :])

    # Find frequency corresponding to maximum STFT magnitude
    max_idx = np.argmax(s_cut, axis=0)
    return f_cut[max_idx] - h_args['f_center']

In [ ]:
def moving_average(signal, window=20):
    x = signal.astype(np.float32, copy=False)
    if window <= 1:
        return x

    c = np.cumsum(np.pad(x, (1, 0), mode='constant'))
    core = (c[window:] - c[:-window]) / window
    pad_left = (window - 1) // 2
    pad_right = window - 1 - pad_left
    return np.pad(core, (pad_left, pad_right), mode='edge')

def _sliding_sum(x, m):
    # returns sum of x inside sliding window of size m
    c = np.cumsum(np.pad(x, (1, 0), mode='constant'))
    return c[m:] - c[:-m]

def calc_cc_array(t, r, eps=1e-12):
    N = len(t)
    sum_xy = fftconvolve(r, t[::-1], mode='valid')
    sum_y = _sliding_sum(r, N)
    sum_y2 = _sliding_sum(r**2, N)
    sum_x = np.sum(t)
    sum_x2 = np.sum(t**2)
    numerator = (N * sum_xy) - (sum_x * sum_y)
    var_x = (N * sum_x2) - (sum_x**2)
    var_y = (N * sum_y2) - (sum_y**2)
    var_y = np.maximum(var_y, 0)
    denom = np.sqrt(var_x) * np.sqrt(var_y)

    return numerator / (denom + eps)

def calc_nm_array(t, r, eps=1e-12):
    N = len(t)
    sum_xy = fftconvolve(r, t[::-1], mode='valid')
    sum_y2 = _sliding_sum(r**2, N)
    sum_x2 = np.sum(t**2)
    numerator = sum_x2 + sum_y2 - (2 * sum_xy)

    return numerator / (sum_y2 + eps)

### Train Dataset Generation Pipeline

The goal of this section is to transform raw audio recordings into a structured dataset suitable for Contrastive Learning. The pipeline consists of four main steps:

1.   **Preprocessing:** Downsampling and splitting raw audio files into consistent chunks.

2.   **Feature Extraction:** Extracting the ENF (Electric Network Frequency) signal from the audio.

3.   **Synchronization:** Using cross-correlation to find ground-truth time overlaps.

4.   **Pairing:** Creating (Anchor, Positive, Negative) triplets for training.



#### 1. Signal Preprocessing (Split & Resample)

Raw audio files are often too large and have high sampling rates (e.g., 44.1kHz).
1.  **Downsampling:** We resample to 400Hz. Since our highest harmonic of interest is 150Hz, 400Hz satisfies the Nyquist theorem ($> 2 \times 150$) while drastically reducing data size.
2.  **Splitting:** We slice long recordings into smaller, overlapping segments (1 to 5 minutes) to augment the dataset.

In [ ]:
# Define paths
target_input_dir = args['train_data_dir'] / "raw_data" / "H1"
reference_input_dir = args['train_data_dir'] / "raw_data" / "H1_ref"
target_output_dir = args['train_data_dir'] / "split_data" / "target"
reference_output_dir = args['train_data_dir'] / "split_data" / "reference"

# Execute Processing
process_directory_parallel(target_input_dir, target_output_dir, is_reference=False)
process_directory_parallel(reference_input_dir, reference_output_dir, is_reference=True)

#### 2. Feature Extraction (ENF Signal)

Here we extract the frequency fluctuation signal from the resampled audio. We use the Short-Time Fourier Transform (STFT) to visualize the frequency content over time, and then identify the frequency bin with the maximum energy around the nominal frequency (50Hz, 100Hz, 150Hz).

Algorithm:

1.  Apply a Bandpass Filter around the target harmonic.

2.  Compute STFT (Spectrogram).

3.  Find the argmax (peak frequency) at every time step.

4.  Subtract the center frequency to get the deviation (ENF).

In [ ]:
input_dir = args['train_data_dir'] / "split_data"
output_dir = args['train_data_dir'] / "enf_data"
signals_count = sum(1 for path in input_dir.rglob('*.npz') if path.is_file())

for harmonic in args['harmonics']:
    f_center = harmonic * args['network_freq']
    window_size = int(args['window_size_sec'] * args['fs'])
    h_args = h_args_parser(harmonic, f_center, args)

    # Get all unique reference signals
    with tqdm(total=signals_count, desc="Extracting ENF signal") as pbar:
        for path in input_dir.rglob('*.npz'):
            if path.is_file():
                if 'ref' in path.name:
                    output_path = output_dir / f"{f_center}hz" / "reference" / path.name
                else:
                    fname_split = path.stem.split('_')
                    new_fname = f"{fname_split[0]}_{int(int(fname_split[1])/400)}_{int(int(fname_split[2])/400)}.npz"
                    output_path = output_dir / f"{f_center}hz" / path.relative_to(input_dir).parent / new_fname
                if not output_path.exists():
                    os.makedirs(output_path.parent, exist_ok=True)
                    with np.load(str(path)) as data:
                        signal = data['signal']
                        signal_enf = extract_enf(signal, args, h_args)
                        np.savez(str(output_path), f=signal_enf)
            pbar.update(1)

#### 3. Ground Truth Alignment (Correlation)

Before training, we must verify the "Ground Truth" timestamps. We use Normalized Cross-Correlation to find exactly where the target signal fits into the reference signal. If the correlation peak occurs at a timestamp matching our metadata (within a tolerance of ALLOWED_LAG_SEC), we consider it a valid labeled sample.

In [ ]:
def time_stamping(target, ref, eps=1e-12):
    t = moving_average(target, 20).astype(np.float32)
    r = moving_average(ref, 20).astype(np.float32)
    return {
        'cc': np.argmax(calc_cc_array(t, r)),
        'nm': np.argmin(calc_nm_array(t, r))
    }

input_dir = args['train_data_dir'] / "enf_data"
freqs = [h * args['network_freq'] for h in args['harmonics']]
enf_subdirs = [input_dir / f"{f}hz" for f in freqs]
signals_count = sum(1 for path in enf_subdirs[0].rglob('*.npz') if path.is_file())

# Create DataFrame
columns = ['recording_length', 'experiment_name', 'split_name']
columns.extend([f"{metric}_t{freqs[i]}_r{freqs[j]}" for metric in ['cc', 'nm'] for i in range(len(args['harmonics'])) for j in range(len(args['harmonics']))])
rows = []

# Load all reference ENF
ref_signals = {path.stem.split('_')[0]: [] for path in (enf_subdirs[0] / "reference").rglob('*.npz')}
for subdir in enf_subdirs:
    for path in (subdir / "reference").rglob('*.npz'):
        with np.load(path) as data:
            ref_name = path.stem.split('_')[0]
            ref_signals[ref_name].append({k: data[k].copy() for k in data})

# Calculate correlation
with tqdm(total=signals_count-len(ref_signals.keys()), desc="Finding estimated time-stamp") as pbar:
    for path in (enf_subdirs[0] / "target").rglob('*.npz'):
        with np.load(path) as target_data:
            ref_name = path.stem.split('_')[0]
            rows.append({
                'recording_length': len(target_data['f']),
                'experiment_name': ref_name,
                'split_name': f"{path.stem.split('_')[1]}_{path.stem.split('_')[2]}"
            })
            gt_time = int(path.name.split("_")[-2])

        for i, subdir in enumerate(enf_subdirs): # Target harmonic
            with np.load(subdir / path.relative_to(enf_subdirs[0])) as target_data:
                for j in range(len(freqs)): # Reference harmonic
                    timestamps = time_stamping(target_data['f'], ref_signals[ref_name][j]['f'])
                    for metric, timestamp in timestamps.items():
                      key = f"{metric}_t{freqs[i]}_r{freqs[j]}"
                      ts_sec = timestamp // 2
                      rows[-1][key] = ts_sec if abs(gt_time - ts_sec) < args['allowed_lag_sec'] else None
        pbar.update(1)

df = pd.DataFrame(rows, columns=columns)
df.to_csv((args['train_data_dir'] / "time_stamp_analysis.csv"), index=False)

#### 4. Final Dataset Construction (Triplet Mining)

This is the most critical step for training the neural network. We organize the data into Triplets (or sets) for InfoNCE/Contrastive Loss. For every "Anchor" signal, we need:

* **Positives:** The correct matching segment in the reference database.

* **Soft Negatives:** A segment from the same reference recording, but at a different time (harder to distinguish than random noise).

* **Hard Negatives:** A segment from a completely different reference recording that happens to have a high correlation score (the "impostors").

In [ ]:
def bucket_samples_by_length(dataset_dict):
    buckets = defaultdict(list)
    for samples in dataset_dict.values():
        for rec_ids, time_indices, signals in samples:
            length = signals.shape[-1]
            buckets[length].append((rec_ids, time_indices, signals))
    return dict(buckets)

def _best_idx_away_from_gt(scores, gt_time, gap, metric):
    n = scores.size
    if n == 0: raise ValueError("Empty correlation array")
    idx = np.arange(n)
    mask = (idx < (gt_time - gap)) | (idx > (gt_time + gap))
    if not np.any(mask):
        return int(np.argmax(scores)) if metric == 'cc' else int(np.argmin(scores))

    if metric == 'cc':
        masked = np.where(mask, scores, -np.inf)
        j = int(np.nanargmax(masked))
        if not np.isfinite(masked[j]): j = int(np.argmax(scores))
    elif metric == 'nm':
        masked = np.where(mask, scores, np.inf)
        j = int(np.nanargmin(masked))
        if not np.isfinite(masked[j]): j = int(np.argmin(scores))

    return j

def get_positive(target, ref_signals, gt_time, exp_name, subdirs):
    m = len(target)
    arr = np.stack(
        [ref_signals[exp_name][i]['f'][gt_time:gt_time + m] for i in range(len(subdirs))],
        axis=0
    ).astype(np.float32, copy=False)
    return torch.from_numpy(arr)

def get_soft_negative(target, ref_signals, gt_time, exp_name, subdirs, metric, gap=30):
    m = len(target)
    ts = []
    tensors = []
    for k in range(len(subdirs)):
        ref_k = ref_signals[exp_name][k]['f']

        if metric == 'cc':
            scores = calc_cc_array(target, ref_k)
        else: # nm
            scores = calc_nm_array(target, ref_k)

        i = _best_idx_away_from_gt(scores, gt_time, gap, metric)

        block = np.stack(
            [ref_signals[exp_name][j]['f'][i:i + m] for j in range(len(subdirs))],
            axis=0
        ).astype(np.float32, copy=False)
        tensors.append(torch.from_numpy(block))
        ts.append(i)
    return ts, tensors

def get_hard_negative(target, ref_signals, gt_time, exp_name, subdirs, metric, allowed_pool=None, gap=30):
    m = len(target)
    out = []

    initial_best = -float('inf') if metric == 'cc' else float('inf')

    for k in range(len(subdirs)):
        best_val = initial_best
        best_entry = None

        for rec_id, per_h in ref_signals.items():
            if rec_id == exp_name: continue
            if allowed_pool is not None and rec_id not in allowed_pool: continue

            ref_k = per_h[k]['f']
            if ref_k.shape[0] <= m: continue

            if metric == 'cc':
                scores = calc_cc_array(target, ref_k)
            else:
                scores = calc_nm_array(target, ref_k)

            i = _best_idx_away_from_gt(scores, gt_time, gap, metric)
            score = float(scores[i])

            if metric == 'cc':
                if score > best_val:
                    best_val = score
                    best_entry = (rec_id, i)
            else: # nm
                if score < best_val:
                    best_val = score
                    best_entry = (rec_id, i)

        if best_entry is None:
            raise ValueError(f"No hard negative found for exp {exp_name} harmonic {k}.")

        choose_rec, i = best_entry
        block = np.stack(
            [ref_signals[choose_rec][j]['f'][i:i + m] for j in range(len(subdirs))],
            axis=0
        ).astype(np.float32, copy=False)
        out.append((choose_rec, i, torch.from_numpy(block)))
    return out


input_dir = args['train_data_dir'] / "enf_data"
freqs = [h * args['network_freq'] for h in args['harmonics']]
enf_subdirs = [input_dir / f"{f}hz" for f in freqs]

# Load csv's
csv_path = args['train_data_dir'] / "time_stamp_analysis.csv"
train_df = pd.read_csv(csv_path)
metric_cols = [c for c in train_df.columns if c.startswith(f"{args['corr_metric']}_t{args['train_target_harmonic']}")]
train_df = train_df.dropna(subset=metric_cols, how='all')

unique_exps = train_df['experiment_name'].unique()
random.seed(47)
unique_exps_list = list(unique_exps)
random.shuffle(unique_exps_list)

# 80/20 Split
split_idx = int(len(unique_exps_list) * 0.8)
train_exps_ids = set(unique_exps_list[:split_idx])
val_exps_ids = set(unique_exps_list[split_idx:])

train_pool_set = {f"{x:03}" for x in train_exps_ids}
val_pool_set = {f"{x:03}" for x in val_exps_ids}

print(f"Total experiments: {len(unique_exps)}")
print(f"Train pool size: {len(train_pool_set)}")
print(f"Val pool size: {len(val_pool_set)}")

train_dict = {}
val_dict = {}

ref_signals = {path.stem.split('_')[0]: [] for path in (enf_subdirs[0] / "reference").rglob('*.npz')}
for subdir in enf_subdirs:
    for path in (subdir / "reference").rglob('*.npz'):
        with np.load(path) as data:
            ref_name = path.stem.split('_')[0]
            f_sm = moving_average(data['f'], 20).astype(np.float32, copy=False)
            ref_signals[ref_name].append({'f': f_sm})

# Create pairs
with tqdm(total=len(train_df), desc="Creating Dataset (Split)") as pbar:
    for _, row in train_df.iterrows():
        gt_time = int(row[metric_cols].mean(skipna=True) * 2)
        exp_name = f"{row['experiment_name']:03}"

        if exp_name in train_pool_set:
            is_train = True
            current_pool = train_pool_set
        elif exp_name in val_pool_set:
            is_train = False
            current_pool = val_pool_set
        else:
            continue

        file_name = f"{exp_name}_{row['split_name']}.npz"

        with np.load((enf_subdirs[1] / "target" / exp_name / file_name)) as target_data:
            target = moving_average(target_data['f'], 20).astype(np.float32, copy=False)

        target_tensor = torch.from_numpy(np.stack([target] * len(enf_subdirs), axis=0))
        pos_tensor = get_positive(target, ref_signals, gt_time, exp_name, enf_subdirs)
        soft_ts, soft_neg_tensors = get_soft_negative(target, ref_signals, gt_time, exp_name, enf_subdirs, args['corr_metric'])

        hard_neg_tensors = get_hard_negative(target, ref_signals, gt_time, exp_name, enf_subdirs, args['corr_metric'], current_pool)

        exp_tensor = torch.tensor((
            int(exp_name), int(exp_name), int(exp_name), int(exp_name), int(exp_name),
            int(hard_neg_tensors[0][0]), int(hard_neg_tensors[1][0]), int(hard_neg_tensors[2][0])
        ), dtype=torch.int32)

        start_idx_tensor = torch.tensor((
            gt_time, gt_time,
            soft_ts[0], soft_ts[1], soft_ts[2],
            hard_neg_tensors[0][1], hard_neg_tensors[1][1], hard_neg_tensors[2][1]
        ), dtype=torch.int32)

        signal_tensor = torch.stack((
            target_tensor,
            pos_tensor,
            soft_neg_tensors[0],
            soft_neg_tensors[1],
            soft_neg_tensors[2],
            hard_neg_tensors[0][2],
            hard_neg_tensors[1][2],
            hard_neg_tensors[2][2],
        ))

        entry = (exp_tensor, start_idx_tensor, signal_tensor)
        if is_train:
            if exp_name not in train_dict: train_dict[exp_name] = []
            train_dict[exp_name].append(entry)
        else:
            if exp_name not in val_dict: val_dict[exp_name] = []
            val_dict[exp_name].append(entry)

        pbar.update(1)

print(f"Processed {len(train_dict)} train experiments and {len(val_dict)} validation experiments.")

train_pairs = bucket_samples_by_length(train_dict)
val_pairs = bucket_samples_by_length(val_dict)

torch.save(train_pairs, args['train_data_dir'] / "train_pairs.pt")
torch.save(val_pairs, args['train_data_dir'] / "val_pairs.pt")

### Test Dataset Generation Pipeline

#### 1. Feature Extraction (ENF Signal)

##### Extract ENF of Raw Recording

In [ ]:
input_dir = args['test_data_dir'] / "split_data"
output_dir = args['test_enf_dir']
signals_count = sum(1 for path in input_dir.rglob('*.npz') if path.is_file())

for harmonic in args['harmonics']:
    f_center = harmonic * args['network_freq']
    window_size = int(args['window_size_sec'] * args['fs'])
    h_args = h_args_parser(harmonic, f_center, args)

    # Get all unique reference signals
    with tqdm(total=signals_count, desc="Extracting ENF signal") as pbar:
        for path in input_dir.rglob('*.npz'):
            if path.is_file():
                if 'ref' in path.name:
                    output_path = output_dir / f"{f_center}hz" / "reference" / path.name
                else:
                    output_path = output_dir / f"{f_center}hz" / path.relative_to(input_dir)
                if not output_path.exists():
                    os.makedirs(output_path.parent, exist_ok=True)
                    with np.load(str(path)) as data:
                        signal = data['data']
                        signal_enf = extract_enf(signal, args, h_args)
                        np.savez(str(output_path), f=signal_enf)
            pbar.update(1)

##### Extract ENF of Recording with AWGN

In [ ]:
def add_gaussian_noise(signal, snr_db):
    signal_power = np.mean(signal ** 2)
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), signal.shape)
    return signal + noise


input_dir = args['test_data_dir'] / "split_data"
signals_count = sum(1 for path in input_dir.rglob('*.npz') if path.is_file())

for snr in args['snr_levels']:
    for harmonic in args['harmonics']:
        f_center = harmonic * args['network_freq']
        window_size = int(args['window_size_sec'] * args['fs'])
        h_args = h_args_parser(harmonic, f_center, args)

        output_dir = args['test_data_dir'] / f"enf_data_{snr}db" / f"{f_center}hz"

        # Get all unique reference signals
        with tqdm(total=signals_count, desc="Extracting ENF signal") as pbar:
            for path in input_dir.rglob('*.npz'):
                if path.is_file():
                    if 'ref' in path.name:
                        output_path = output_dir / "reference" / path.name
                        if not output_path.exists():
                            os.makedirs(output_path.parent, exist_ok=True)
                            with np.load(str(path)) as data:
                                signal = data['data']
                                signal_enf = extract_enf(signal, args, h_args)
                                np.savez(str(output_path), f=signal_enf)
                    else:
                        output_path = output_dir / path.relative_to(input_dir)
                        if not output_path.exists():
                            os.makedirs(output_path.parent, exist_ok=True)
                            with np.load(str(path)) as data:
                                signal = data['data']
                                noisy_signal = add_gaussian_noise(signal, snr)
                                signal_enf = extract_enf(noisy_signal, args, h_args)
                                np.savez(str(output_path), f=signal_enf)
                pbar.update(1)

#### 2. Dataset Construction

In [ ]:
def sliding_windows(lst, size, jump):
    windows = [lst[i:i+size] for i in range(0, len(lst) - size + 1, jump)]
    return [moving_average(win) for win in windows]

raw_data_dir = args['test_data_dir'] / "split_data"
enf_subdirs = [args['test_enf_dir'] / f"{f}hz" for f in args['harmonics_freq']]
delta_t = 0.5

exp_dict, ref_signals = {}, {}
for exp_dir in (raw_data_dir / '5 min').iterdir():
    if exp_dir.is_dir():
        exp_dict[exp_dir.name] = []
        ref_fname = [fname for fname in os.listdir(raw_data_dir / '5 min' / exp_dir) if fname.startswith('ref')][0]
        ref_signals[exp_dir.name] = {}
        for subdir in enf_subdirs:
            with np.load((subdir / 'reference' / ref_fname)) as ref_data:
                ref_signals[exp_dir.name][subdir.name] = {k: ref_data[k].copy() for k in ref_data}

for t in args['segment_dur']:
    curr_len = int(t * 60 / delta_t)
    for exp_name, ref_signal in ref_signals.items():
        windows = []
        for subdir in enf_subdirs:
            ref = ref_signal[subdir.name]['f']
            windows.append(sliding_window_view(ref, curr_len))
        target_tensor = torch.tensor(np.stack(windows, axis=0), dtype=torch.float32)
        exp_dict[exp_name].append(target_tensor)

torch.save(exp_dict, args['test_data_dir'] / "test_refs.pt")

In [ ]:
def pad_or_crop_1d(arr, target_len):
    result = torch.zeros([arr.shape[0], target_len], dtype=arr.dtype)
    length = min(arr.shape[1], target_len)
    result[:,:length] = arr[:,:length]
    return result

raw_data_dir = args['test_data_dir'] / "split_data"
enf_dir = args['test_data_dir'] / f"enf_data"
enf_subdirs = [enf_dir / f"{f}hz" for f in args['harmonics_freq']]
delta_t = 0.5

test_df = pd.read_csv(args['test_data_dir'] / "time_stamp_02df_cc.csv")

# Create array for each experiment and reference signal per experiment
exp_dict, ref_signals = {}, {}
for exp_dir in (raw_data_dir / '5 min').iterdir():
    if exp_dir.is_dir():
        # Init array per experiment
        exp_dict[exp_dir.name] = []
        # Save ref signal per harmony per experiment
        ref_fname = [fname for fname in os.listdir(raw_data_dir / '5 min' / exp_dir) if fname.startswith('ref')][0]
        ref_signals[exp_dir.name] = {}
        for subdir in enf_subdirs:
            with np.load((subdir / 'reference' / ref_fname)) as ref_data:
                ref_signals[exp_dir.name][subdir.name] = {k: ref_data[k].copy() for k in ref_data}

# Create pairs
with tqdm(total=len(test_df), desc="Creating Dataset") as pbar:
    for t, rec_len in zip(args['segment_dur'], sorted(test_df['recording_length'].unique())):
        curr_len = int(t * 60 / delta_t)
        min_df = test_df[test_df['recording_length'] == rec_len]
        for _, row in min_df.iterrows():
            gt_time = int(int(row['file_name'].split("_")[-2]) / 0.5)
            # Experiment name
            exp_name = [k for k in ref_signals.keys() if k in row['experiment_name']][0]
            # Create samples
            with np.load(str(enf_subdirs[1] / rec_len / row['experiment_name'] / row['file_name'])) as target_data:
                target = moving_average(target_data['f'], 20).astype(np.float32, copy=False)
                target_tensor = torch.tensor(np.stack([target] * len(enf_subdirs), axis=0), dtype=torch.float32)
                target_tensor = pad_or_crop_1d(target_tensor, curr_len)
                exp_dict[exp_name].append((target_tensor, gt_time))
            pbar.update(1)
torch.save(exp_dict, args['test_data_dir'] / f"test_targets.pt")

Construct for each AWGN

In [ ]:
def pad_or_crop_1d(arr, target_len):
    result = torch.zeros([arr.shape[0], target_len], dtype=arr.dtype)
    length = min(arr.shape[1], target_len)
    result[:,:length] = arr[:,:length]
    return result

for snr in args['snr_levels']:
    raw_data_dir = args['test_data_dir'] / "split_data"
    enf_dir = args['test_data_dir'] / f"enf_data_{snr}db"
    enf_subdirs = [enf_dir / f"{f}hz" for f in args['harmonics_freq']]
    delta_t = 0.5

    test_df = pd.read_csv(args['test_data_dir'] / "time_stamp_02df_cc.csv")


    # Create array for each experiment and reference signal per experiment
    exp_dict, ref_signals = {}, {}
    for exp_dir in (raw_data_dir / '5 min').iterdir():
        if exp_dir.is_dir():
            # Init array per experiment
            exp_dict[exp_dir.name] = []
            # Save ref signal per harmony per experiment
            ref_fname = [fname for fname in os.listdir(raw_data_dir / '5 min' / exp_dir) if fname.startswith('ref')][0]
            ref_signals[exp_dir.name] = {}
            for subdir in enf_subdirs:
                with np.load((subdir / 'reference' / ref_fname)) as ref_data:
                    ref_signals[exp_dir.name][subdir.name] = {k: ref_data[k].copy() for k in ref_data}

    # Create pairs
    with tqdm(total=len(test_df), desc="Creating Dataset") as pbar:
        for t, rec_len in zip(args['segment_dur'], sorted(test_df['recording_length'].unique())):
            curr_len = int(t * 60 / delta_t)
            min_df = test_df[test_df['recording_length'] == rec_len]
            for _, row in min_df.iterrows():
                gt_time = int(int(row['file_name'].split("_")[-2]) / 0.5)
                # Experiment name
                exp_name = [k for k in ref_signals.keys() if k in row['experiment_name']][0]
                # Create samples
                with np.load(str(enf_subdirs[1] / rec_len / row['experiment_name'] / row['file_name'])) as target_data:
                    target = moving_average(target_data['f'], 20).astype(np.float32, copy=False)
                    target_tensor = torch.tensor(np.stack([target] * len(enf_subdirs), axis=0), dtype=torch.float32)
                    target_tensor = pad_or_crop_1d(target_tensor, curr_len)
                    exp_dict[exp_name].append((target_tensor, gt_time))
                pbar.update(1)
    torch.save(exp_dict, project_dir / "data" / "datasets" / f"test_pairs_{snr}db.pt")

## Training

In [ ]:
encoder_train_dataset = torch.load(args['train_data_dir'] / "train_pairs.pt")
encoder_val_dataset = torch.load(args['train_data_dir'] / "val_pairs.pt")

encoder = TwoStageTCNEncoder()
encoder = train_encoder(encoder, encoder_train_dataset, encoder_val_dataset, epochs=20, batch_size=128, device=device)

In [ ]:
path = "./data/models/encoder.pth"
torch.save(encoder.state_dict(), path)

____________________________________________
# Train Fusion Layer


## Dataset

### Create dataset

In [ ]:
def embed_train_set(model, train_dataset, batch_size, device):
    def new_dict(keys):
        return {k: [] for k in keys}

    def fill_temp_dict(temp_dict, keys, values):
        for k, v in zip(keys, values):
            temp_dict[k].append(v)

    def create_final_dict(temp_dict, keys):
        return {L: {k: torch.cat(v[k], dim=0) for k in keys}
                for L, v in temp_dict.items()}

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
    use_amp = torch.cuda.is_available()
    amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16

    keys = ("exp_id", "pos", "embs", "sig")
    model = model.to(device).eval()
    loader = BucketedBatchLoader(train_dataset, batch_size=batch_size, drop_last=True)
    target_dict = defaultdict(lambda: new_dict(keys))
    ref_dict = defaultdict(lambda: new_dict(keys))

    with torch.no_grad():
        for rec_ids, time_indices, batch in loader:
            batch = batch.to(device, non_blocking=True)
            rec_ids = rec_ids.to(device, non_blocking=True)
            time_indices = time_indices.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=use_amp, dtype=amp_dtype):
                embeddings = model(batch)
                embeddings = F.normalize(embeddings, dim=1)

            L = batch.shape[-1]
            fill_temp_dict(target_dict[L], keys, (rec_ids[::8], time_indices[::8], embeddings[::8], batch[::8]))
            fill_temp_dict(ref_dict[L], keys, (rec_ids[1::8], time_indices[1::8], embeddings[1::8], batch[1::8]))
    return create_final_dict(target_dict, keys), create_final_dict(ref_dict, keys)

def create_fusion_dataset(test_dict, ref_dict, k=50, device='cpu', time_window=60):
    scores, pos_masks, L_list = [], [], []

    for L in sorted(test_dict.keys()):
        # Load reference data
        ref_sig = ref_dict[L]['sig'].to(device, copy=False)
        ref_embs = ref_dict[L]['embs'].to(device, copy=False)
        ref_exp = ref_dict[L]['exp_id'].to(device, copy=False)
        ref_pos = ref_dict[L]['pos'].to(device, copy=False)
        # Load test data
        test_sig = test_dict[L]['sig'].to(device, copy=False)
        test_embs = test_dict[L]['embs'].to(device, copy=False)
        test_exp = test_dict[L]['exp_id'].to(device, copy=False)
        gt_times = test_dict[L]['pos'].to(device, copy=False)

        # Embedding scores
        emb_scores = test_embs @ ref_embs.T

        pos_mask, topk_idx = compute_top_k(k, emb_scores, ref_exp, ref_pos, test_exp, gt_times, time_window)

        # Signal corr scores
        cc_scores, nm_scores = [], []
        for ch in range(3):
            cc_scores.append(calc_cc_score(test_sig[:, ch, :], ref_sig[:, ch, :]))
            nm_scores.append(-calc_nm_score(test_sig[:, ch, :], ref_sig[:, ch, :]))

        rows = torch.arange(emb_scores.shape[0]).unsqueeze(1)

        topk_emb_scores = emb_scores[rows, topk_idx]
        topk_cc_scores = [cc[rows, topk_idx] for cc in cc_scores]
        topk_nm_scores = [nm[rows, topk_idx] for nm in nm_scores]

        scores.append(torch.stack(([topk_emb_scores] + topk_cc_scores), dim=-1))
        pos_masks.append(pos_mask)
        L_list.extend([L] * len(pos_mask))

    temp_scores = torch.cat(scores, dim=0)
    temp_mask = torch.cat(pos_masks, dim=0)
    non_empty_idx = [i for i, t in enumerate(temp_mask) if t.any()] # samples that top_k scores have a positive
    print(f"Non-empty: {len(non_empty_idx) / len(temp_mask):.1%}")

    return {'scores': temp_scores[non_empty_idx],
            'pos_mask': temp_mask[non_empty_idx],
            'L': [L_list[i] for i in non_empty_idx]}, L_list

### DataLoader

In [ ]:
class ProjectionBucketedBatchLoader:
    def __init__(self, data, batch_size, drop_last=True, seed=47):
        self.data = data
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.seed = seed
        self.epoch = 0

        self.buckets = {L: [] for L in set(data['L'])}
        for i, L in enumerate(data['L']):
            self.buckets[L].append((data['scores'][i], data['pos_mask'][i], data['L'][i]))

    def __iter__(self):
        all_batches = []
        random.seed(self.seed + self.epoch)  # Different shuffle each epoch

        for length, samples in self.buckets.items():
            samples = samples.copy()
            random.shuffle(samples)

            # Form batches
            for i in range(0, len(samples), self.batch_size):
                batch = samples[i:i + self.batch_size]
                if len(batch) == self.batch_size or not self.drop_last:
                    features = torch.stack([scores for scores, *_ in batch], dim=0)
                    masks = torch.stack([mask for _, mask, _ in batch], dim=0)
                    L = torch.tensor([l for *_, l in batch])
                    all_batches.append((features, masks, L))

        random.shuffle(all_batches)  # Shuffle batch order across buckets
        self.epoch += 1
        return iter(all_batches)

    def __len__(self):
        total_batches = 0
        for samples in self.buckets.values():
            n = len(samples)
            if self.drop_last:
                total_batches += n // self.batch_size
            else:
                total_batches += (n + self.batch_size - 1) // self.batch_size
        return total_batches

## Training

In [ ]:
encoder_train_dataset = torch.load(args['train_data_dir'] / "train_pairs.pt")
encoder_val_dataset = torch.load(args['train_data_dir'] / "val_pairs.pt")

train_target_set, train_ref_set = embed_train_set(encoder, encoder_train_dataset, batch_size=128, device=device)
fusion_train_dataset, _ = create_fusion_dataset(train_target_set, train_ref_set, k=100, device=device)

val_target_set, val_ref_set = embed_train_set(encoder, encoder_val_dataset, batch_size=128, device=device)
fusion_val_dataset, _ = create_fusion_dataset(val_target_set, val_ref_set, k=100, device=device)

In [ ]:
fusion_layer = train_fusion_length_bucketed(
    fusion_train_dataset, fusion_val_dataset,
    batch_size=512, epochs=10, temperature=0.07, lr=0.5, weight_decay=1e-4, drop_last=False)

In [ ]:
path = "./data/models/fusion.pth"
torch.save(fusion_layer.state_dict(), path)

____________________________________________
# Results


## Compute Datasets for Evaluating

In [ ]:
encoder = TwoStageTCNEncoder()
encoder.load_state_dict(torch.load("./data/models/encoder.pth", map_location=device))
encoder = encoder.to(device)
fusion_layer = LinearFusion(in_dim=4)
fusion_layer.load_state_dict(torch.load("./data/models/fusion.pth", map_location=device))
fusion_layer = fusion_layer.to(device)

### Embed Datasets for Testing

In [ ]:
def moving_average(arr, window=20):
    B, C, L = arr.shape
    x = arr.numpy().reshape(B*C, L)
    c = np.cumsum(np.pad(x, ((0, 0), (1, 0)), mode="constant"), axis=1)
    core = (c[:, window:] - c[:, :-window]) / window
    pad_left = (window - 1) // 2
    pad_right = window - 1 - pad_left
    out = np.pad(core, ((0, 0), (pad_left, pad_right)), mode="edge")

    return torch.from_numpy(out).reshape(B, C, L).to(torch.float32)

def embed_test_ref(data, model, device, batch_size):
    all_embeddings = []
    ref_indices = []  # (experiment, window_index)

    model.eval()
    with torch.no_grad():
        for exp_name, window_tensors in data.items():
            for window_index, tensor_group in enumerate(window_tensors):
                tensor_group = tensor_group.permute(1, 0, 2)
                tensor_group = moving_average(tensor_group)
                num_windows = tensor_group.shape[0]
                signal_len = tensor_group.shape[-1]

                for i in range(0, num_windows, batch_size):
                    batch = tensor_group[i:i+batch_size].to(device)
                    emb = model(batch)
                    emb = F.normalize(emb, dim=1)

                    all_embeddings.append(emb)
                    ref_indices.extend([(exp_name, signal_len, i + j, batch[j]) for j in range(emb.size(0))])

    ref_embeddings = torch.cat(all_embeddings, dim=0)

    temp_dict = {}
    exp_to_id = {e: i for i, e in enumerate(sorted({x[0] for x in ref_indices}))}
    for (exp_name, ref_len, start_index, signal), emb in zip(ref_indices, ref_embeddings):
        if ref_len not in temp_dict:
            temp_dict[ref_len] = {'exp_id': [], 'pos': [], 'embs': [], 'sig': []}
        temp_dict[ref_len]['exp_id'].append(exp_to_id[exp_name])
        temp_dict[ref_len]['pos'].append(start_index)
        temp_dict[ref_len]['embs'].append(emb)
        temp_dict[ref_len]['sig'].append(signal)

    return {k: {'exp_id': torch.tensor(v['exp_id']),
                'pos': torch.tensor(v['pos']),
                'embs': torch.stack(v['embs']),
                'sig': torch.stack(v['sig'])
                } for k, v in temp_dict.items()}

def test_by_len(dataset_dict):
    buckets = {t*120: {} for t in np.arange(1, 5.5, 0.5)}
    for length in buckets.keys():
        for exp_name, test_signals in dataset_dict.items():
            buckets[length][exp_name] = [(test_tensor, gt_time) for test_tensor, gt_time in test_signals if test_tensor.shape[-1] == length]

    return buckets

def embed_test_target(test_dataset, model):
    temp_dict = {}
    exp_to_id = {e: i for i, e in enumerate(sorted({x for x in test_dataset.keys()}))}

    test_buckets = test_by_len(test_dataset)
    model.eval()
    with torch.no_grad():
      for L, data in test_buckets.items():
        for exp_name, test_signals in data.items():
          for test_tensor, gt_time in test_signals:
              test_tensor = test_tensor.unsqueeze(0).to(device)
              test_len = test_tensor.shape[-1]
              if test_len not in temp_dict:
                temp_dict[test_len] = {'exp_id': [], 'pos': [], 'embs': [], 'sig': []}

              test_emb = model(test_tensor)
              test_emb = F.normalize(test_emb, dim=1)

              temp_dict[test_len]['exp_id'].append(exp_to_id[exp_name])
              temp_dict[test_len]['pos'].append(gt_time)
              temp_dict[test_len]['embs'].append(test_emb.squeeze_(0))
              temp_dict[test_len]['sig'].append(test_tensor.squeeze_(0))

    return {k: {'exp_id': torch.tensor(v['exp_id']),
                'pos': torch.tensor(v['pos']),
                'embs': torch.stack(v['embs']),
                'sig': torch.stack(v['sig'])
                } for k, v in temp_dict.items()}

### Evaluate Functions

In [ ]:
def compare_tests(test_dict, ref_dict, device='cpu'):
    results = {'emb': [], 'cc': [], 'nm': []}
    stats = {
        'emb': {'all': [0, 0], 'short': [0, 0]},
        'cc':  {'all': [0, 0], 'short': [0, 0]},
        'nm':  {'all': [0, 0], 'short': [0, 0]}
    }

    for L in sorted(test_dict.keys()):
        # Load reference data
        ref_sig = ref_dict[L]['sig'].to(device, copy=False)
        ref_embs = ref_dict[L]['embs'].to(device, copy=False)
        ref_exp = ref_dict[L]['exp_id'].to(device, copy=False)
        ref_pos = ref_dict[L]['pos'].to(device, copy=False)
        # Load test data
        test_sig = test_dict[L]['sig'].to(device, copy=False)
        test_embs = test_dict[L]['embs'].to(device, copy=False)
        test_exp = test_dict[L]['exp_id'].to(device, copy=False)
        gt_times = test_dict[L]['pos'].to(device, copy=False)

        num_samples = test_exp.shape[0]
        is_short = int(L) <= 240

        # --- Embedding scores ---
        emb_sim = test_embs @ ref_embs.T
        emb_correct, _ = compute_top_k(1, emb_sim, ref_exp, ref_pos, test_exp, gt_times)
        emb_correct = emb_correct.squeeze(1)

        n_correct = emb_correct.sum().item()
        results['emb'].append(n_correct / num_samples * 100)

        stats['emb']['all'][0] += n_correct
        stats['emb']['all'][1] += num_samples
        if is_short:
            stats['emb']['short'][0] += n_correct
            stats['emb']['short'][1] += num_samples

        # --- Correlation scores ---
        for c_test in range(3):
            for c_ref in range(3):
                freq_test = (c_test + 1) * 50
                freq_ref = (c_ref + 1) * 50

                # --- CC ---
                cc = calc_cc_score(test_sig[:, c_test, :], ref_sig[:, c_ref, :])
                sig_correct, _  = compute_top_k(1, cc, ref_exp, ref_pos, test_exp, gt_times)
                sig_correct = sig_correct.squeeze(1)
                n_correct_cc = sig_correct.sum().item()

                if c_test == c_ref and c_test == 2:
                    results['cc'].append(n_correct_cc / num_samples * 100)
                    stats['cc']['all'][0] += n_correct_cc
                    stats['cc']['all'][1] += num_samples
                    if is_short:
                        stats['cc']['short'][0] += n_correct_cc
                        stats['cc']['short'][1] += num_samples

                # --- NM ---
                nm = -calc_nm_score(test_sig[:, c_test, :], ref_sig[:, c_ref, :])
                nm_correct, _  = compute_top_k(1, nm, ref_exp, ref_pos, test_exp, gt_times)
                nm_correct = nm_correct.squeeze(1)
                n_correct_nm = nm_correct.sum().item()

                if (c_test == c_ref and c_test == 1 and L <= 240) or (c_test == c_ref and c_test == 2 and L > 240):
                    results['nm'].append(n_correct_nm / num_samples * 100)
                    stats['nm']['all'][0] += n_correct_nm
                    stats['nm']['all'][1] += num_samples
                    if is_short:
                        stats['nm']['short'][0] += n_correct_nm
                        stats['nm']['short'][1] += num_samples

    # Print Aggregate Results
    print("\n" + "="*40)
    print("AGGREGATE RESULTS")
    print("="*40)

    for metric in ['emb', 'cc', 'nm']:
        # Total
        total_corr = stats[metric]['all'][0]
        total_cnt = stats[metric]['all'][1]
        total_acc = (total_corr / total_cnt * 100) if total_cnt > 0 else 0.0

        # Short (<= 2 min)
        short_corr = stats[metric]['short'][0]
        short_cnt = stats[metric]['short'][1]
        short_acc = (short_corr / short_cnt * 100) if short_cnt > 0 else 0.0

        print(f"Metric: {metric.upper()}")
        print(f"  Total Avg Acc: {total_acc:.2f}% ({total_corr}/{total_cnt})")
        print(f"  Short (<=2) Acc: {short_acc:.2f}% ({short_corr}/{short_cnt})")
        print("-" * 20)

    return results

def get_acc(model, data, full_data, batch_size=256, device='cpu'):
    results = []
    phi = data['scores'].to(device)
    pos_mask = data['pos_mask'].to(device)

    scores = []
    total_correct = 0
    total_count = 0
    short_correct = 0
    short_count = 0
    SHORT_THRESHOLD = 2

    model.eval()
    with torch.no_grad():
        for i in range(0, phi.shape[0], batch_size):
            scores.append(model(phi[i:i+batch_size]))

    scores = torch.cat(scores, dim=0)
    top1_idx = scores.argmax(dim=-1)

    counter = {L: 0 for L in set(data['L'])}

    for i in range(len(pos_mask)):
        if pos_mask[i][top1_idx[i]]:
            counter[data['L'][i]] += 1

    for L in sorted(counter.keys()):
        n_correct = counter[L]
        n_total = full_data.count(L)

        results.append(n_correct / n_total * 100)

        total_correct += n_correct
        total_count += n_total

        if L <= 240:
            short_correct += n_correct
            short_count += n_total

    # --- Print Aggregate Results ---
    print("\n" + "="*40)
    print("MODEL ACCURACY RESULTS")
    print("="*40)

    # Total
    total_acc = (total_correct / total_count * 100) if total_count > 0 else 0.0
    print(f"  Total Avg Acc: {total_acc:.2f}% ({total_correct}/{total_count})")

    # Short (<= 2 min)
    short_acc = (short_correct / short_count * 100) if short_count > 0 else 0.0
    print(f"  Short (<=2) Acc: {short_acc:.2f}% ({short_correct}/{short_count})")
    print("-" * 20)

    return results

## Results for Raw Signals

In [ ]:
test_ref_data = torch.load(args['test_data_dir'] / 'test_refs.pt')
test_ref_emb = embed_test_ref(test_ref_data, encoder, device=device, batch_size=128) # ref_d

test_target_data = torch.load(args['test_data_dir'] / 'test_targets.pt')
test_target_emb = embed_test_target(test_target_data, encoder)

results = compare_tests(test_target_emb, test_ref_emb, device=device)

test_dataset, full_L = create_fusion_dataset(test_target_emb, test_ref_emb, k=5000, device=device)
results['fus'] = get_acc(fusion_layer, test_dataset, full_L, batch_size=512, device=device)

In [ ]:
plt.figure(figsize=(8, 6))

# Plotting CC
plt.plot(args['segment_dur'], results['cc'], '-o', label='Baseline Method (CC)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.1))

# Plotting NM
plt.plot(args['segment_dur'], results['nm'], '-o', label='Baseline Method (NM)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.35))

# Plotting DL (Deep Learning model)
plt.plot(args['segment_dur'], results['emb'], '-o', label='Our Method (Encoder)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.6))
plt.plot(args['segment_dur'], results['fus'], '-o', label='Our Method (Encoder + Linear Fusion)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.9))

# Labels and title
plt.xlabel('Target duration [min]', fontsize=12)
plt.xlim(1.0,5.0)
plt.ylabel('Success rate [%]', fontsize=12)
plt.ylim(0, 100)
plt.grid(True)
plt.legend(loc='lower right', fontsize='x-large')
plt.tight_layout()
plt.show()


# Results for Signals with AWGN

In [ ]:
encoder = TwoStageTCNEncoder()
encoder.load_state_dict(torch.load("./data/models/encoder.pth", map_location=device))
encoder = encoder.to(device)
fusion_layer = LinearFusion(in_dim=4)
fusion_layer.load_state_dict(torch.load("./data/models/fusion.pth", map_location=device))
fusion_layer = fusion_layer.to(device)

test_ref_data = torch.load(args['test_data_dir'] / 'test_refs.pt')
test_ref_emb = embed_test_ref(test_ref_data, encoder, device=device, batch_size=128)

In [ ]:
results = {snr: {} for snr in args['snr_levels']}
for snr in args['snr_levels']:
    print(snr)
    test_target_data = torch.load(args['test_data_dir'] / f"test_pairs_{snr}db.pt")
    test_target_emb = embed_test_target(test_target_data, encoder)
    test_dataset, full_L = create_fusion_dataset(test_target_emb, test_ref_emb, k=5000, device=device)

    results[snr] = compare_tests(test_target_emb, test_ref_emb, device=device)
    results[snr]['fus'] = get_acc(fusion_layer, test_dataset, full_L, batch_size=512, device=device)
    for k, v in results[snr].items():
        print(f"{k}: {', '.join(f'{x:.2f}' for x in v)}")

In [ ]:
results_plot = {l: {'emb': [], 'cc': [], 'nm': [], 'fus': []} for l in args['segment_dur']}
for i, l in enumerate(args['segment_dur']):
  for snr in args['snr_levels']:
    for k, v in results[snr].items():
      results_plot[l][k].append(v[i])

for i, l in enumerate(args['segment_dur']):
    plt.figure(figsize=(8, 6))
    plt.title(f"Recordings of Length {l} min")

    # Plotting CC
    plt.plot(args['snr_levels'], results_plot[l]['cc'], '-o', label='Baseline Method (CC)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.1))
    # Plotting NM
    plt.plot(args['snr_levels'], results_plot[l]['nm'], '-o', label='Baseline Method (NM)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.35))
    # Plotting DL (Deep Learning model)
    plt.plot(args['snr_levels'], results_plot[l]['emb'], '-o', label='Our Method (Encoder)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.6))
    plt.plot(args['snr_levels'], results_plot[l]['fus'], '-o', label='Our Method (Encoder + Linear Fusion)', linewidth=2, markersize=6, zorder=10, clip_on=False, color=plt.cm.viridis(0.9))

    # Labels and title
    plt.xlabel('SNR [db]', fontsize=12)
    plt.xlim(-20,20)
    plt.ylabel('Success rate [%]', fontsize=12)
    plt.ylim(0, 100)
    plt.grid(True)
    plt.legend(loc='lower right', fontsize='x-large')
    plt.tight_layout()
    plt.show()